In [1]:
import pandas as pd
import numpy as np
import warnings
import re
import datetime
from difflib import SequenceMatcher
warnings.filterwarnings("ignore")

In [2]:
# 原始数据路径
input_path = r"./原始数据-盖锡咨询-中国项目数据库历史数据-202508预测 .xlsx"
# 清洗后结果路径
output_path1 = r"./预测每月拆分地区集中式202509v2.xlsx"
output_path2 = r"./预测每月拆分地区分布式202509v2.xlsx"
output_path3 = r"./预测每月拆分地区类型202509v2.xlsx"

df_data = pd.read_excel(input_path, sheet_name = '国内中标')

In [3]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27678 entries, 0 to 27677
Data columns (total 61 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   招标日期                          25244 non-null  object        
 1   中标日期                          27677 non-null  datetime64[ns]
 2   中标月份                          27677 non-null  float64       
 3   标案状态                          16343 non-null  object        
 4   项目编号                          15396 non-null  object        
 5   项目名称                          27677 non-null  object        
 6   项目类型                          27675 non-null  object        
 7   招标人                           27643 non-null  object        
 8   项目地址-省                        27469 non-null  object        
 9   项目地址-市                        26927 non-null  object        
 10  中标人                           27618 non-null  object        
 11  中标公司简称                      

In [4]:
df1 = df_data[(df_data['招标内容'] == 'EPC总承包') | (df_data['招标内容'] == '组件采购')]
df1 = df1[(df1['项目类型'] == '光伏治沙') | (df1['项目类型'] == '地面电站')| (df1['项目类型'] == '农光互补')| (df1['项目类型'] == '风光互补')
          | (df1['项目类型'] == '年度采购') | (df1['项目类型'] == '集中采购')| (df1['项目类型'] == '框架采购')
          | (df1['项目类型'] == '平价上网') | (df1['项目类型'] == '领跑者')| (df1['项目类型'] == '年度集采')]#集中式
df1['项目规模MW'] = df1['项目规模MW'].fillna(0)


df2 = df_data[(df_data['招标内容'] == 'EPC总承包') | (df_data['招标内容'] == '组件采购')]
df2 = df2[(df2['项目类型'] == '分布式') | (df2['项目类型'] == '扶贫')]#分布式
df2['项目规模MW'] = df2['项目规模MW'].fillna(0)

df3 = df_data[(df_data['招标内容'] == 'EPC总承包') | (df_data['招标内容'] == '组件采购')]
df3['项目规模MW'] = df3['项目规模MW'].fillna(0)

In [5]:
cat1 = df1['项目地址-省']
cat2 = df2['项目地址-省']
cat3 = df3['项目类型']

df_output1 = pd.DataFrame(data = cat1,columns = ['项目地址-省',
                                          '202101','202102','202103','202104','202105','202106',
                                          '202107','202108','202109','202110','202111','202112',
                                          '202201','202202','202203','202204','202205','202206',
                                          '202207','202208','202209','202210','202211','202212',
                                          '202301','202302','202303','202304','202305','202306',
                                          '202307','202308','202309','202310','202311','202312',
                                          '202401','202402','202403','202404','202405','202406',
                                          '202407','202408','202409','202410','202411','202412',
                                          '202501','202502','202503','202504','202505','202506',
                                          '202507','202508','202509','202510','202511','202512',
                                          '202601','202602','202603','202604','202605','202606',
                                          '202607','202608','202609','202610','202611','202612',
                                          '202701','202702','202703','202704','202705','202706',
                                          '202707','202710',
                                          '202801','202803','202804','202805','202806','202811',
                                          '202901','202903','202908',
                                          '203010','203012',
                                          '203101',
                                          '203510','203512',
                                          '203707','203711',
                                          '203911',
                                          '204402','204410',
                                          '204504',
                                          '205006',
                                          '205411'])
df_output2 = pd.DataFrame(data = cat2,columns = ['项目地址-省',
                                          '202101','202102','202103','202104','202105','202106',
                                          '202107','202108','202109','202110','202111','202112',
                                          '202201','202202','202203','202204','202205','202206',
                                          '202207','202208','202209','202210','202211','202212',
                                          '202301','202302','202303','202304','202305','202306',
                                          '202307','202308','202309','202310','202311','202312',
                                          '202401','202402','202403','202404','202405','202406',
                                          '202407','202408','202409','202410','202411','202412',
                                          '202501','202502','202503','202504','202505','202506',
                                          '202507','202508','202509','202510','202511','202512',
                                          '202601','202602','202603','202604','202605','202606',
                                          '202607','202608','202609','202610','202611','202612',
                                          '202701','202702','202703','202704','202705','202706',
                                          '202707','202710',
                                          '202801','202803','202804','202805','202806','202811',
                                          '202901','202903','202908',
                                          '203010','203012',
                                          '203101',
                                          '203510','203512',
                                          '203707','203711',
                                          '203911',
                                          '204402','204410',
                                          '204504',
                                          '205006',
                                          '205411'])
df_output3 = pd.DataFrame(data = cat3,columns = ['项目类型',
                                          '202101','202102','202103','202104','202105','202106',
                                          '202107','202108','202109','202110','202111','202112',
                                          '202201','202202','202203','202204','202205','202206',
                                          '202207','202208','202209','202210','202211','202212',
                                          '202301','202302','202303','202304','202305','202306',
                                          '202307','202308','202309','202310','202311','202312',
                                          '202401','202402','202403','202404','202405','202406',
                                          '202407','202408','202409','202410','202411','202412',
                                          '202501','202502','202503','202504','202505','202506',
                                          '202507','202508','202509','202510','202511','202512',
                                          '202601','202602','202603','202604','202605','202606',
                                          '202607','202608','202609','202610','202611','202612',
                                          '202701','202702','202703','202704','202705','202706',
                                          '202707','202710',
                                          '202801','202803','202804','202805','202806','202811',
                                          '202901','202903','202908',
                                          '203010','203012',
                                          '203101',
                                          '203510','203512',
                                          '203707','203711',
                                          '203911',
                                          '204402','204410',
                                          '204504',
                                          '205006',
                                          '205411'])
print(df_output3["项目类型"].unique())
df1['交付月份'] = df1['交付月份'].fillna(0)
for i in df1.index.tolist():
    df1.loc[i,'中标月份'] = str(int(df1.loc[i,'中标月份']))
    df1.loc[i,'交付月份'] = str(int(df1.loc[i,'交付月份']))
    if (df1.loc[i,'中标月份'] in df_output1.columns) and (df1.loc[i,'交付月份'] in df_output1.columns):
        month_win = df1.loc[i,'中标月份']
        month_fin = df1.loc[i,'交付月份']
        mw_per_month = df1.loc[i,'项目规模MW'] / len(df_output1.loc[i,month_win:month_fin])
        df_output1.loc[i,month_win:month_fin] = mw_per_month

df2['交付月份'] = df2['交付月份'].fillna(0)
for i in df2.index.tolist():
    df2.loc[i,'中标月份'] = str(int(df2.loc[i,'中标月份']))
    df2.loc[i,'交付月份'] = str(int(df2.loc[i,'交付月份']))
    if (df2.loc[i,'中标月份'] in df_output2.columns) and (df2.loc[i,'交付月份'] in df_output2.columns):
        month_win = df2.loc[i,'中标月份']
        month_fin = df2.loc[i,'交付月份']
        mw_per_month = df2.loc[i,'项目规模MW'] / len(df_output2.loc[i,month_win:month_fin])
        df_output2.loc[i,month_win:month_fin] = mw_per_month

df3['交付月份'] = df3['交付月份'].fillna(0)
for i in df3.index.tolist():
    df3.loc[i,'中标月份'] = str(int(df3.loc[i,'中标月份']))
    df3.loc[i,'交付月份'] = str(int(df3.loc[i,'交付月份']))
    if (df3.loc[i,'中标月份'] in df_output3.columns) and (df3.loc[i,'交付月份'] in df_output3.columns):
        month_win = df3.loc[i,'中标月份']
        month_fin = df3.loc[i,'交付月份']
        mw_per_month = df3.loc[i,'项目规模MW'] / len(df_output3.loc[i,month_win:month_fin])
        df_output3.loc[i,month_win:month_fin] = mw_per_month
        print(df_output3.loc[i,month_win:month_fin])

['分布式' '农光互补' '地面电站' '领跑者' '特高压' '年度集采' '竞价' 'BIPV' '框架采购' '光伏治沙' '海上光伏'
 nan '平价上网' '扶贫' 0 '项目招标' '光储项目招标']
202101    0.018091
202102    0.018091
202103    0.018091
202104    0.018091
Name: 2668, dtype: object
202101    0.709
202102    0.709
Name: 2669, dtype: object
202101    0.066667
202102    0.066667
202103    0.066667
Name: 2672, dtype: object
202101    0.0152
Name: 2674, dtype: object
202101    0.009077
202102    0.009077
202103    0.009077
202104    0.009077
202105    0.009077
Name: 2676, dtype: object
202101    0.229333
202102    0.229333
202103    0.229333
Name: 2680, dtype: object
202101    4.004983
202102    4.004983
202103    4.004983
202104    4.004983
202105    4.004983
202106    4.004983
Name: 2681, dtype: object
202101    0.031387
202102    0.031387
Name: 2687, dtype: object
202101    0.0
202102    0.0
202103    0.0
202104    0.0
202105    0.0
202106    0.0
202107    0.0
202108    0.0
202109    0.0
202110    0.0
202111    0.0
202112    0.0
202201    0.0
202202    0.0
2

In [6]:
df_large = df_output1.groupby('项目地址-省').sum()
df_small = df_output2.groupby('项目地址-省').sum()
df_cat = df_output3.groupby('项目类型').sum()

In [7]:
df_large.to_excel(output_path1)

In [8]:
df_small.to_excel(output_path2)

In [9]:
df_cat.to_excel(output_path3)

In [9]:
df_large.to_excel(output_path1)
df_small.to_excel(output_path2)
df_cat.to_excel(output_path3)